In [0]:
from pyspark.sql.functions import col, to_date, to_timestamp,round
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text( "bronze_catalog", "dbr_dev")
dbutils.widgets.text("bronze_schema","artemzharkov10_bronze")
dbutils.widgets.text("silver_catalog","dbr_dev")
dbutils.widgets.text( "silver_schema", "artemzharkov10_silver")

BRONZE_CATALOG = dbutils.widgets.get("bronze_catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")

SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

In [0]:
bronze_stream = spark.readStream.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.stream_bitcoin_bronze")
silver_chekpoint = f"/Volumes/{SILVER_CATALOG}/{SILVER_SCHEMA}/checkpoint/stream_bitcoin_silver"



Creating a buffer batch in which duplicates are checked and values are transform to the appropriate type. This is done to prevent too much data from being stored in the classer's memory at the same time. To achieve this, we implement micro-batching. 

In [0]:
# spark give argument to a function (data set (batch), batch ID (used for older tables that do not support the MERGE operation; for such tables, the batch ID is required))
def silver_batch(micro_batch_df, batch_id):
    clean_batch = (
        micro_batch_df
        .withColumnRenamed("Date","TradeDate")
        .dropDuplicates(["TradeDate"])
        .filter(col("_rescued_data").isNull())
        .filter(col("Market_cap").isNotNull())
        .withColumn("TradeDate",to_timestamp(col("TradeDate"),"yyyy-MM-dd HH:mm:ss"))
        .withColumn("Close",round(col("Close").cast("Double"),2))
        .withColumn("High",round(col("High").cast("Double"),2))
        .withColumn("Low",round(col("Low").cast("Double"),2))
        .withColumn("Open",round(col("Open").cast("Double"),2))
        .withColumn("Volume",round(col("Volume").cast("Double"),4))
        .withColumn("Market_cap",round(col("Market_cap").cast("Double")))
        )     
    
    silver_table_path = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.stream_bitcoin_silver"

    if(spark.catalog.tableExists(silver_table_path)):

        silver_table = DeltaTable.forName(spark,silver_table_path) 

        (silver_table.alias("table") # for avoiding simililar name of column
        .merge(
            clean_batch.alias("batch"),
            "table.TradeDate = batch.TradeDate" # like Primary key via make merge/update
            )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()  
        .execute()
        )
    else:
        clean_batch.write.format("delta").saveAsTable(silver_table_path)


In [0]:

query = (
    bronze_stream
    .writeStream
    .foreachBatch(silver_batch)
    .option("checkpointLocation",silver_chekpoint)
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination() # end call after finishig process new data (.option(availableNow = true))

In [0]:

# silver_table_path = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.stream_bitcoin_silver"
# silver_df = spark.read.table(silver_table_path)
# display(silver_df)